# Capstone — Prioritizing SEO Content for Human Review with Historical Search Signals

**Research paper companion notebook** · **Author:** Ali Almasri  
**Research lane:** Refresh / Content Opportunity Scoring

This notebook mirrors the research paper. It tests whether March 2026 search-performance signals rank content by April 2026 click-through rate (CTR) more usefully than a position-peer baseline, and separately audits performance when entire clients are held out.

**Decision supported:** help a human reviewer decide what content to inspect first. This is decision support—not a causal analysis of SEO changes and not an attempt to infer Google's ranking algorithm.

> **Privacy:** Do not display client/content identifiers, client names, content URLs, private queries, credentials, or raw warehouse rows. Store the Hugging Face token in a private environment variable or Colab secret; never commit it.

## 1. Question

**Research question:** Do historical search-performance features produce a more useful ranking of next-month CTR than a simple position-peer baseline, and does the model retain a positive ranking association when entire clients are held out?

The decision is narrow: rank content candidates for human review using information available in March. The system does not automatically change pages or promise traffic growth.

## 2. Data

- **Dataset:** `FlyRank/internship-warehouse`
- **Release build:** `flyrank_pseudonymized_warehouse_release_v20260703` (exported 2026-07-03)
- **Primary table:** `fact_content_daily_performance` (reported size: 78,835,655 rows)
- **Freshness:** daily facts through 2026-06-30, with a three-day freshness cutoff
- **Feature window:** March 1–31, 2026
- **Outcome window:** April 1–30, 2026
- **Reported March-to-April join:** 331,436 matched rows
- **Recorded model-ready population:** 137,742 rows after requiring at least 10 April impressions and complete required fields

The label is `future_ctr = April clicks / April impressions`. The minimum-April-impression threshold reduces extreme low-denominator CTR values, but uses future-window information to define the retrospective evaluation population. It is **not** an appropriate live eligibility rule at March decision time.

**March-only predictors:** `gsc_impressions`, `gsc_clicks`, `observed_ctr`, `avg_position`, and one-hot position group (`1-3`, `4-10`, `11-20`, `21-50`, `50+`).

**Excluded from model features:** April clicks/impressions/CTR, `ctr_gap`, `baseline_score`, and identifiers. IDs are used only for joining and client-grouped validation. Public artifacts should show aggregate metrics only.

### Connect to the gated dataset

Run in an environment authorized to access the FlyRank Hugging Face dataset. In Colab, add `HF_TOKEN` through **Secrets** and grant notebook access, or set it as an environment variable. Never paste a token into a code cell.

In [ ]:
import os
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add it as a private environment variable or Colab secret. "
        "Do not hard-code the token in this notebook."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)
rel = "hf://datasets/FlyRank/internship-warehouse"
march_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
april_path = f"{rel}/fact_content_daily_performance/month=2026-04/*.parquet"
print("Connected. Analysis windows: March 2026 features → April 2026 outcome.")

### Build March features and April outcome

Aggregate daily records by pseudonymous client/content keys, compute March CTR and impression-weighted average position, then join the windows. Identifiers remain internal to joins and validation logic.

In [ ]:
march_df = con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS gsc_impressions,
       SUM(gsc_clicks) AS gsc_clicks,
       SUM(gsc_sum_position) AS gsc_sum_position
FROM read_parquet('{march_path}')
WHERE gsc_impressions > 0
GROUP BY client_hash_id, content_hash_id
""").df()

march_df["observed_ctr"] = march_df["gsc_clicks"] / march_df["gsc_impressions"].replace(0, np.nan)
march_df["avg_position"] = march_df["gsc_sum_position"] / march_df["gsc_impressions"].replace(0, np.nan)

def assign_position_group(position):
    if pd.isna(position): return np.nan
    if position <= 3: return "1-3"
    if position <= 10: return "4-10"
    if position <= 20: return "11-20"
    if position <= 50: return "21-50"
    return "50+"

march_df["position_group"] = march_df["avg_position"].apply(assign_position_group)

april_df = con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS future_impressions,
       SUM(gsc_clicks) AS future_clicks
FROM read_parquet('{april_path}')
GROUP BY client_hash_id, content_hash_id
""").df()
april_df["future_ctr"] = april_df["future_clicks"] / april_df["future_impressions"].replace(0, np.nan)

model_df = march_df.merge(april_df, on=["client_hash_id", "content_hash_id"], how="inner")
joined_rows = len(model_df)
model_df = model_df.loc[
    (model_df["future_impressions"] >= 10) &
    model_df["future_ctr"].notna() &
    model_df["observed_ctr"].notna() &
    model_df["avg_position"].notna() &
    model_df["position_group"].notna()
].copy()

print(f"Matched rows before threshold: {joined_rows:,}")
print(f"Model-ready rows: {len(model_df):,}")
print("No identifiers are displayed.")

### Data-quality and label checks

The target is sparse and right-skewed. These checks show aggregate statistics only.

In [ ]:
display(model_df["future_ctr"].describe(percentiles=[.25, .50, .75, .99]).to_frame("future_ctr"))

plt.figure(figsize=(8, 4.8))
upper = model_df["future_ctr"].quantile(.99)
plt.hist(model_df["future_ctr"].clip(upper=upper), bins=50)
plt.title("April future CTR distribution (clipped at 99th percentile for display)")
plt.xlabel("April future CTR")
plt.ylabel("Number of content records")
plt.tight_layout()
plt.show()

## 3. Methodology

### Baseline: position-peer priority score

For each March position group, calculate exposure-weighted expected CTR as `total clicks / total impressions`. Compute the positive gap between expected CTR and observed March CTR. Normalize the gap and `log1p(impressions)`, then combine them:

`baseline_score = 0.7 × gap_score + 0.3 × impression_score`

This is a transparent priority score, **not a forecast**. The benchmark is weighted aggregate CTR, not median page-level CTR.

### Model: Random Forest regressor

The model predicts April CTR from March-only features. Configuration: 100 trees, maximum depth 12, minimum leaf size 20, random seed 42, `n_jobs=-1`. The primary comparison uses an 80/20 random row split (`random_state=42`). A separate client-held-out split tests robustness to unseen clients.

In [ ]:
feature_cols = ["gsc_impressions", "gsc_clicks", "observed_ctr", "avg_position", "position_group"]
X = pd.get_dummies(model_df[feature_cols], columns=["position_group"], drop_first=False)
y = model_df["future_ctr"].copy()

forbidden = {
    "future_impressions", "future_clicks", "future_ctr",
    "ctr_gap", "baseline_score", "client_hash_id", "content_hash_id"
}
leakage_found = forbidden.intersection(set(X.columns))
assert not leakage_found, f"Potential leakage fields found: {leakage_found}"
assert len(X) == len(y) == len(model_df)
assert X.isna().sum().sum() == 0 and y.isna().sum() == 0
print("Feature matrix:", X.shape)
print("Leakage guard: PASS — no future outcomes, derived baseline fields, or IDs in X.")

### Rebuild the baseline using March only

The benchmark and priority score are built from March observations. Both methods will be evaluated on the exact same validation records. Raw baseline scores and predicted CTR are not directly comparable because they have different meanings.

In [ ]:
baseline_source = march_df.copy()
position_benchmark = (
    baseline_source.groupby("position_group", as_index=False)
    .agg(total_clicks=("gsc_clicks", "sum"),
         total_impressions=("gsc_impressions", "sum"))
)
position_benchmark["expected_ctr"] = (
    position_benchmark["total_clicks"] /
    position_benchmark["total_impressions"].replace(0, np.nan)
)
baseline_source = baseline_source.merge(
    position_benchmark[["position_group", "expected_ctr"]],
    on="position_group", how="left"
)
baseline_source["ctr_gap"] = baseline_source["expected_ctr"] - baseline_source["observed_ctr"]
positive_gap = baseline_source["ctr_gap"].clip(lower=0)
gap_max = positive_gap.max()
baseline_source["gap_score"] = positive_gap / gap_max if gap_max > 0 else 0.0
baseline_source["impression_score"] = np.log1p(baseline_source["gsc_impressions"])
imp_max = baseline_source["impression_score"].max()
baseline_source["impression_score"] = (
    baseline_source["impression_score"] / imp_max if imp_max > 0 else 0.0
)
baseline_source["baseline_score"] = (
    .7 * baseline_source["gap_score"] + .3 * baseline_source["impression_score"]
)
print("Baseline built from March features only.")

### Main same-split comparison

The random row split can place content from the same client in training and validation. This is an acknowledged limitation; the separate client-held-out audit below addresses it.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)
model = RandomForestRegressor(
    n_estimators=100, max_depth=12, min_samples_leaf=20,
    random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
ml_pred = model.predict(X_valid)

valid_model = model_df.loc[X_valid.index].copy()
valid_model["ml_prediction"] = ml_pred
valid_baseline = valid_model[
    ["client_hash_id", "content_hash_id", "future_ctr", "future_impressions", "ml_prediction"]
].merge(
    baseline_source[["client_hash_id", "content_hash_id", "baseline_score"]],
    on=["client_hash_id", "content_hash_id"], how="left", validate="one_to_one"
)
assert len(valid_baseline) == len(y_valid)
assert valid_baseline["baseline_score"].notna().all()

rf_mae = mean_absolute_error(y_valid, ml_pred)
rf_rmse = mean_squared_error(y_valid, ml_pred) ** 0.5
rf_r2 = r2_score(y_valid, ml_pred)
baseline_spearman = spearmanr(valid_baseline["baseline_score"], valid_baseline["future_ctr"]).statistic
rf_spearman = spearmanr(valid_baseline["ml_prediction"], valid_baseline["future_ctr"]).statistic

metrics_table = pd.DataFrame([
    {"Method": "Position-peer baseline", "Spearman": baseline_spearman,
     "MAE": np.nan, "RMSE": np.nan, "R²": np.nan},
    {"Method": "Random Forest", "Spearman": rf_spearman,
     "MAE": rf_mae, "RMSE": rf_rmse, "R²": rf_r2},
])
display(metrics_table.round(6))
print(f"Same-split validation rows: {len(valid_model):,}")

### Top-K ranking comparison

Compare the observed mean April CTR among each method's highest-ranked validation records. These are retrospective selected-group outcomes, not causal uplift estimates or guaranteed gains.

In [ ]:
def top_k_mean_future_ctr(frame, score_col, k):
    return frame.sort_values(score_col, ascending=False).head(k)["future_ctr"].mean()

topk_rows = []
for k in [20, 50, 100, 500]:
    topk_rows.append({
        "K": k,
        "Baseline mean future CTR": top_k_mean_future_ctr(valid_baseline, "baseline_score", k),
        "Random Forest mean future CTR": top_k_mean_future_ctr(valid_baseline, "ml_prediction", k),
    })
topk_table = pd.DataFrame(topk_rows)
display(pd.DataFrame({
    "K": topk_table["K"],
    "Baseline mean future CTR (%)": topk_table["Baseline mean future CTR"] * 100,
    "Random Forest mean future CTR (%)": topk_table["Random Forest mean future CTR"] * 100,
}).round(4))

### Client-held-out generalization audit

`GroupShuffleSplit` holds out entire pseudonymous clients. No client should occur in both training and validation. This is a separate design and is not a direct baseline-vs-model comparison on the same grouped split.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, grouped_valid_idx = next(
    gss.split(X, y, groups=model_df["client_hash_id"])
)
train_clients = set(model_df.iloc[train_idx]["client_hash_id"])
valid_clients = set(model_df.iloc[grouped_valid_idx]["client_hash_id"])
client_overlap = train_clients.intersection(valid_clients)
assert not client_overlap, "Client leakage: overlap across grouped split."

X_train_g, X_valid_g = X.iloc[train_idx], X.iloc[grouped_valid_idx]
y_train_g, y_valid_g = y.iloc[train_idx], y.iloc[grouped_valid_idx]
grouped_model = RandomForestRegressor(
    n_estimators=100, max_depth=12, min_samples_leaf=20,
    random_state=42, n_jobs=-1
)
grouped_model.fit(X_train_g, y_train_g)
grouped_pred = grouped_model.predict(X_valid_g)

grouped_results = pd.DataFrame([{
    "Validation design": "Client-held-out",
    "Validation rows": len(y_valid_g),
    "Held-out clients": len(valid_clients),
    "Spearman": spearmanr(y_valid_g, grouped_pred).statistic,
    "MAE": mean_absolute_error(y_valid_g, grouped_pred),
    "RMSE": mean_squared_error(y_valid_g, grouped_pred) ** 0.5,
    "R²": r2_score(y_valid_g, grouped_pred),
}])
display(grouped_results.round(6))
print("Client overlap:", len(client_overlap))

### Leakage and validation audit

The model uses March-only features. April fields define the target and retrospective evaluation population, not the feature matrix. The April-impression filter is a known selection limitation even though it is not a predictive feature.

In [ ]:
leakage_audit = pd.DataFrame([
    {"Check": "Predictors are derived from March", "Status": "PASS"},
    {"Check": "April clicks/impressions/CTR excluded from X", "Status": "PASS"},
    {"Check": "Baseline score and CTR gap excluded from X", "Status": "PASS"},
    {"Check": "Identifiers excluded from X", "Status": "PASS"},
    {"Check": "No client overlap in grouped validation", "Status": "PASS"},
    {"Check": "Future-impression filter caveat documented", "Status": "DISCLOSED"},
])
display(leakage_audit)

## 4. Results (vs baseline)

The recorded same-split experiment used **27,549 validation rows** and reported:

- Spearman rank correlation: baseline **−0.250472**, Random Forest **0.343931**
- Random Forest MAE **0.002425**, RMSE **0.006364**, R² **0.140516**

Recorded mean future CTR among top-ranked validation records:

| K | Position-peer baseline | Random Forest |
|---:|---:|---:|
| 20 | 0.0578% | 6.0012% |
| 50 | 0.0610% | 3.7988% |
| 100 | 0.0487% | 2.9967% |
| 500 | 0.1091% | 1.4608% |

The positive Spearman correlation suggests that higher model predictions tended to correspond to higher observed April CTR in this validation sample. The modest R² means most variation remains unexplained. CTR is sparse and heavy-tailed, so top-K values—especially top 20—may be sensitive to a small number of records. Re-run the notebook to regenerate the tables and charts; investigate any differences before updating the paper.

### Artifacts for the paper

The following figures are aggregate-only and do not contain identifiers or content URLs.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
names = ["Position-peer baseline", "Random Forest"]
vals = [baseline_spearman, rf_spearman]
bars = ax.bar(names, vals)
ax.axhline(0, linewidth=.8)
ax.set_ylabel("Spearman correlation with April CTR")
ax.set_title("Ranking association on the same validation split")
ax.bar_label(bars, fmt="%.3f", padding=3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
x = np.arange(len(topk_table))
width = .36
ax.bar(x - width/2, topk_table["Baseline mean future CTR"] * 100,
       width, label="Position-peer baseline")
ax.bar(x + width/2, topk_table["Random Forest mean future CTR"] * 100,
       width, label="Random Forest")
ax.set_xticks(x, [str(k) for k in topk_table["K"]])
ax.set_xlabel("Top-K records")
ax.set_ylabel("Mean observed April CTR (%)")
ax.set_title("Observed future CTR among top-ranked records")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
designs = ["Random row split", "Client-held-out split"]
vals = [rf_spearman, grouped_results.loc[0, "Spearman"]]
bars = ax.bar(designs, vals)
ax.axhline(0, linewidth=.8)
ax.set_ylabel("Random Forest Spearman correlation")
ax.set_title("Ranking association across validation designs")
ax.bar_label(bars, fmt="%.3f", padding=3)
plt.tight_layout()
plt.show()

## 5. Limitations and honest framing

1. **Observational, not causal:** no evidence that CTR, position, or any feature causes future performance; no estimate of uplift from editing a page.
2. **No claim about Google's algorithm:** the model ranks observed outcomes and does not reveal how Google ranks content.
3. **Sparse, heavy-tailed CTR:** low-volume observations can be noisy even after filtering.
4. **Evaluation selection:** the 10-April-impression filter uses future information and limits the population to which the reported metrics apply.
5. **Limited features:** query intent, SERP features, device, geography, content semantics, and seasonality are not directly modeled.
6. **Uncertain generalization:** the client-held-out result is one split with nine held-out clients; repeated client and temporal validation are needed.
7. **Different score meanings:** the baseline is a priority score and the Random Forest predicts CTR; compare rankings and top-K outcomes, not raw score magnitudes.
8. **Human review required:** a high score is a candidate for investigation, not a confirmed defect or an automatic instruction to edit.

## 6. Ranked recommendations — action playbook

1. **Review high-visibility content with CTR below position peers.** Inspect query intent and search-result presentation before changing titles or snippets. A gap is a prompt to investigate, not proof of a snippet problem.
2. **Inspect content around positions 11–20 when March impressions are meaningful.** Check intent match, completeness, internal links, and position stability.
3. **Investigate model–observation discrepancies.** Treat them as trend or data-quality review candidates, not estimates of potential uplift.
4. **Use evidence thresholds cautiously.** The operational queue treats at least 1,000 impressions as higher observed visibility and 100–299 as more limited evidence. These are workflow heuristics, not confidence intervals.
5. **Keep human approval and record interventions.** Log the hypothesis, change, date, and outcome window; avoid untracked simultaneous edits.
6. **Use cooldowns and explicit monitoring windows.** Account for seasonality and query-mix changes before reassessing.
7. **Monitor model quality and queue integrity.** Track ranking metrics, errors, benchmark coverage, missing values, duplicate keys, score validity, reason-code distribution, and outcomes by client and position group.

### Queue artifact

The companion action-playbook workflow's recorded export contains **23,496 rows and 19 columns**, ranks 1–23,496, no missing values in audited columns, nonnegative priority scores, and no missing position-peer benchmark values. Operational queue eligibility uses at least 100 **March** impressions. This differs from the retrospective 10-**April**-impression evaluation filter.

## 7. Artifacts the paper embeds

This notebook regenerates:
- the April target-distribution diagnostic;
- the same-split baseline-vs-model Spearman chart;
- the top-K observed future CTR chart;
- the separate validation-design Spearman chart;
- aggregate result tables and leakage checks;
- the ranked recommendations above.

Do not add individual client/content records to public tables or figures.

In [ ]:
paper_metrics = pd.DataFrame([
    {"Method": "Position-peer baseline", "Spearman": baseline_spearman},
    {"Method": "Random Forest", "Spearman": rf_spearman,
     "MAE": rf_mae, "RMSE": rf_rmse, "R²": rf_r2},
])
paper_topk = pd.DataFrame({
    "K": topk_table["K"],
    "Baseline mean future CTR (%)": topk_table["Baseline mean future CTR"] * 100,
    "Random Forest mean future CTR (%)": topk_table["Random Forest mean future CTR"] * 100,
})
print("Same-split comparison")
display(paper_metrics.round(6))
print("Top-K comparison (percent)")
display(paper_topk.round(4))
print("Separate client-held-out audit")
display(grouped_results.round(6))

## Reproducibility

- **Repository:** [AliAlmasri7/FlyRank-ML](https://github.com/AliAlmasri7/FlyRank-ML)
- **Dataset:** [FlyRank internship warehouse](https://huggingface.co/datasets/FlyRank/internship-warehouse)
- **Question and task framing:** `work/notebooks/w01_research_question.ipynb`, `work/notebooks/w02_ml_task_framing.ipynb`
- **Data contract and leakage audit:** `work/notebooks/w03_data_contract.ipynb`, `work/notebooks/w03_feature_leakage_check.ipynb`
- **Baseline and model:** `work/notebooks/w04_baseline_score.ipynb`, `work/notebooks/w05_model.ipynb`
- **Validation and action playbook:** `work/notebooks/w06_validation_audit.ipynb`, `work/notebooks/w07_action_playbook.ipynb`
- **This notebook:** `work/notebooks/capstone.ipynb`

Provide authorized dataset access through a private `HF_TOKEN`, then run top-to-bottom. If regenerated values differ from the recorded results, investigate release version, feature construction, row ordering, and split before revising the paper. Never commit credentials or publish raw warehouse rows.

## Acknowledgments and data credit

**Built on the [FlyRank ML Internship dataset](https://flyrank.ai).**

The analysis uses pseudonymized warehouse data. Interpretations are the author's and do not imply causal effects or guaranteed performance.

## Conclusion

On the recorded same-split experiment, the Random Forest had a more positive ranking association with April CTR than the position-peer baseline. It also retained a positive ranking association in a separate client-held-out evaluation. These findings support testing the model as a human-in-the-loop triage aid, with continued validation and monitoring. They do not show that acting on ranked pages will improve performance or that results generalize to every client and period.